# LAB: Agents that Speak MCP Fluently (three servers)

**Ayhan Demir** · Ironhack AI Consulting & Integration Bootcamp · Module 4, Week 4

## Objective

In `mcp_introduction.ipynb` we connected an agent to **one remote** MCP server. In this lab we connect **three very different servers**:

- 🌐 a **Fetch server** that runs **on our own laptop** and reads public web pages
- 🐙 a **GitHub server** that runs **on GitHub's computers** and reads code repositories
- 📚 the **LangChain Docs server** that runs **on LangChain's computers** and searches its documentation


## Use case: agent framework shortlist for a client RFP

A client asks which agent framework to shortlist. A consultant needs **facts, not opinions**. We compare **LangGraph** and the **OpenAI Agents SDK**, and give every server **one job**, so each fact has one clear source:

| Fact | Server | Tool or resource |
|---|---|---|
| License and latest release | 🐙 GitHub | `get_file_contents` (LICENSE), `get_latest_release` |
| What LangGraph is for | 📚 LangChain Docs | The docs search tools |
| What the OpenAI Agents SDK is for | 🌐 Fetch | `fetch` (its docs site is not part of the LangChain docs) |
| When to choose LangGraph | 📚 LangChain Docs | The skill guide **resource** |

## 🗺️ Steps we will follow

| # | Step | What happens |
|---|------|--------------|
| 1 | **Setup and installation** | Install packages, load the OpenAI key and the GitHub token, check `uvx` |
| 2 | **Connect to MCP servers** | One client per server: Fetch (**stdio**), GitHub (**HTTP** + token), LangChain Docs (**HTTP**, no token). List tools and resources |
| 3 | **Load MCP tools into LangChain** | Load the tools of all three servers and leave out the one tool that writes |
| 4 | **Create an agent with MCP tools** | One agent per server, and read every tool call behind the answer |
| 5 | **Access MCP resources** | Give an agent read-only background: the LangChain skill guide |
| 6 | **Build a complete MCP-enabled agent** | Three clients, one agent with all tools **and** the resource, one RFP question that needs all three servers |
| 7 | Optional: MCP vs direct API | Not done (optional) |
| 8 | Optional: practical example | Not done (optional) |
| ✅ | **`lab_proof.md`** | The entry point for the reviewer |

**Sources**
- GitHub repositories: `langchain-ai/langgraph` and `openai/openai-agents-python`
- LangChain Docs MCP server: `https://docs.langchain.com/mcp` (tools + the skill guide resource)
- Documentation page read with Fetch: `https://openai.github.io/openai-agents-python/`

---
## Step 1: Setup and Installation

1. `.env` file next to this notebook (never committed):

| Server | Add to `.env` |
|---|---|
| All | `OPENAI_API_KEY=sk-...` |
| Fetch | Nothing |
| LangChain Docs | Nothing |
| **GitHub** | `GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...` (fine-grained, *Public repositories (read-only)*) |


In [1]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your git-ignored .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")   # only needed for GitHub
# uvx starts the Fetch server: from Homebrew, or from this environment (pip install uv)
UVX = shutil.which("uvx") or os.path.join(os.path.dirname(sys.executable), "uvx")

print("OpenAI key:", "loaded" if os.getenv("OPENAI_API_KEY") else "missing")
print("uvx:", "found" if os.path.exists(UVX) else "NOT found: install uv, then restart the kernel")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing: needed for GitHub")
print("LangChain Docs: remote MCP endpoint; no token required")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


OpenAI key: loaded
uvx: found
GitHub token: loaded
LangChain Docs: remote MCP endpoint; no token required


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")
    result = await handler(request)   # the real tool runs on the server
    # AFTER: what the server sent back
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result

---
## Step 2: Connect to MCP Servers

**Goal:** **one client per server**, then test each connection by listing its tools and resources.

```

In [4]:
# Fetch client: local server, started by uvx
fetch_client = MultiServerMCPClient(
    {
        "fetch": {
            "transport": "stdio",
            "command": UVX,
            "args": ["mcp-server-fetch@2026.10.24"],
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("Fetch client configured")

Fetch client configured


### 2b. GitHub client (HTTP + token)

GitHub hosts its official MCP server at `https://api.githubcopilot.com/mcp/`. We send **`headers`** with every request:

| Header | Value | Why |
|---|---|---|
| `Authorization` | `f"Bearer {GITHUB_TOKEN}"` | Our ID card. Without a valid token GitHub rejects the call (*401* or *400*) |
| `X-MCP-Toolsets` | `"repos,issues"` | Only load these groups of tools |
| `X-MCP-Readonly` | `"true"` | Only read tools: the agent cannot change anything on GitHub |

The npm package `@modelcontextprotocol/server-github` is **deprecated**, so we use GitHub's official server.

In [5]:
# GitHub client: remote server, token in the headers (read from .env, never printed)
github_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}",
                "X-MCP-Toolsets": "repos,issues",
                "X-MCP-Readonly": "true",
            },
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("GitHub client configured")

GitHub client configured


### 2c. LangChain Docs client (HTTP, no token)

The same config as in `mcp_introduction.ipynb`, now **with the interceptor**

In [6]:
# LangChain Docs client: remote server, public, no token
docs_client = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp",
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("LangChain Docs client configured")

LangChain Docs client configured


### 2d. Test all three connections

For **each** client: count the tools with `get_tools()`, and try `get_resources()` inside a `try/except`.

In [7]:
clients = {"fetch": fetch_client, "github": github_client, "langchain-docs": docs_client}

for name, client in clients.items():
    tools = await client.get_tools()
    print(f"{name}: {len(tools)} tools")
    try:
        resources = await client.get_resources()
        print(f"{name}: {len(resources)} resources")
    except Exception as error:
        print(f"{name}: no resources ({type(error).__name__})")

fetch: 1 tools
fetch: 0 resources
github: 19 tools
github: 1 resources
langchain-docs: 3 tools
langchain-docs: 1 resources


**Done when:** all three clients return a tool count, and we know which servers offer resources and which don't.


---
## Step 3: Load MCP Tools into LangChain

**Goal:** turn the MCP tools into LangChain tools and check they are accessible. `get_tools()` already converts every MCP tool into a LangChain tool.


In [8]:
fetch_tools = await fetch_client.get_tools()
github_tools = await github_client.get_tools()
docs_tools = await docs_client.get_tools()

# Read-only agent: leave out the one tool that sends something
docs_tools = [t for t in docs_tools if t.name != "submit_feedback"]

print("Fetch tools: ", [t.name for t in fetch_tools])
print("GitHub tools:", [t.name for t in github_tools])
print("Docs tools:  ", [t.name for t in docs_tools])

Fetch tools:  ['fetch']
GitHub tools: ['get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories']
Docs tools:   ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']


**Done when:** we can see the name of every tool from all three servers.

---
## Step 4: Create an Agent with MCP Tools

**Goal:** one agent per server that answers from its source, with a receipt. Each question is one the LLM **cannot** answer from memory, so it must call a tool.

In [9]:
# Fetch agent: answers from a documentation page
fetch_agent = create_agent(
    model=llm,
    tools=fetch_tools,
    system_prompt=(
        "You read web pages with the fetch tool. Always fetch the page before you answer, "
        "never guess, and name the URL you used."
    ),
)

question = "Read https://openai.github.io/openai-agents-python/. In one or two sentences: what does the OpenAI Agents SDK say it is for?"

fetch_result = await fetch_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", fetch_result["messages"][-1].content)

[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/'}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
<p>OpenAI Agents SDK</p>

Skip to content

# OpenAI Agents SDK

The OpenAI Agents SDK enab...
[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/', 'start_index': 5000}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
esume state | Results |
| Build a low-latency voice agent with gpt-realtime-2.1 | Realtime...

Answer: The OpenAI Agents SDK is designed to enable the creation of agentic AI applications in a lightweight and user-friendly manner, providing essential tools for building agents that can perform tasks, delegate work, and ensure input/output validation. It aims to simplify the development process while allowing for complex interactions and real-world applications. 

You can find more information at [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/).

In [10]:
# GitHub agent: answers from a repository
github_agent = create_agent(
    model=llm,
    tools=github_tools,
    system_prompt=(
        "You read public GitHub repositories with the GitHub tools. Never guess: every fact "
        "must come from a tool result. Name the repository and the file or tool you used."
    ),
)

question = (
    "For the GitHub repository langchain-ai/langgraph: which license does it use "
    "(read the LICENSE file), and what is its latest release?"
)

github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", github_result["messages"][-1].content)

[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG]    returned: {"tag_name":"1.2.18","target_commitish":"3521f08e21a7d89c28331191374bc85c11ecb6e2","name":"langgraph==1.2.18","body":"Changes since 1.2.17\n\n* releas...
[LOG]    returned: successfully downloaded text file (SHA: fc0602feecdd6748623c852ab534e1ca612673c7) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

Answer: The GitHub repository **langchain-ai/langgraph** uses the **MIT License**. The license text states that permission is granted to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the software, with the condition that the copyright notice and permission notice are included in all copies or substantial portions of the software.

The latest release of the repository is **version 1.2.18**, which was published on **October 26, 2026

In [11]:
# Docs agent: answers from the LangChain documentation
docs_agent = create_agent(
    model=llm,
    tools=docs_tools,
    system_prompt=(
        "You answer questions about LangChain and LangGraph with the LangChain docs tools. "
        "Always search the docs before you answer, never guess, and name the page link you used."
    ),
)

question = "Search the LangChain docs: in one or two sentences, what is LangGraph for?"

docs_result = await docs_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", docs_result["messages"][-1].content)

[LOG] langchain-docs -> search_docs_by_lang_chain  {'query': 'LangGraph'}
[LOG]    returned: Title: LangGraph overview
Link: https://docs.langchain.com/oss/javascript/langgraph/overview
Page: oss/javascript/langgraph/overview
Content: ## <Icon...

Answer: LangGraph is a framework designed for creating Pregel applications, allowing users to define a graph of nodes and edges that can be compiled into a Pregel application. It simplifies the development of complex workflows by providing high-level APIs for managing state and processing data in a structured manner. For more details, you can refer to the [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview).


---
## Step 5: Access MCP Resources

**Goal:** give an agent read-only **background knowledge** instead of tools.

| | Tools | Resources |
|---|---|---|
| **What** | Actions: *do* something | Read-only data: *know* something |
| **Who decides** | The **agent** calls them | **Our code** fetches them and hands them over |

Step 2d showed which servers offer resources. The **LangChain Docs** client from Step 2 offers one: the LangChain skill guide. It fits the RFP: the guide says **when to pick LangGraph**. No new client is needed.

In [12]:
# Resource from the LangChain Docs client (Step 2c)
resources = await docs_client.get_resources()
guide_text = resources[0].as_string()
print(f"{len(resources)} resource(s); the guide has {len(guide_text)} characters")

# Agent with NO tools: the guide text is its only source
guide_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "Answer ONLY from the guide below. If the guide does not say it, answer 'Not in the guide'.\n\n"
        f"GUIDE:\n{guide_text}"
    ),
)

question = "According to the guide, when should a team use LangGraph instead of LangChain's create_agent?"

guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content
print("\nAnswer:", answer)

# Evidence: a resource has no tool call, so we check key words from the answer in the resource text
for word in ["deterministic", "human-in-the-loop"]:
    print(f"'{word}': in answer = {word in answer.lower()}, in guide = {word in guide_text.lower()}")

1 resource(s); the guide has 11252 characters

Answer: A team should use LangGraph instead of LangChain's create_agent when they need low-level control for complex workflows that mix deterministic logic with LLM-driven steps. LangGraph is suitable for scenarios requiring human-in-the-loop processes and deterministic steps.
'deterministic': in answer = True, in guide = True
'human-in-the-loop': in answer = True, in guide = True


---
## Step 6: Build a Complete MCP-Enabled Agent

**Goal:** one agent that uses the tools of **three servers**, through **three clients**, **plus** the resource from Step 5, to answer the RFP question that needs **all three**. An agent doesn't care where a tool comes from: it just gets a **list of tools**.

The system prompt gives each server **one job** (the table at the top). With 22 tools and two ways to read LangChain pages (`fetch` or the docs search), a clear job per server keeps the trace readable.

In [13]:
# Three clients -> one tool list (reuse the tools from Step 3)
all_tools = fetch_tools + github_tools + docs_tools

print(f"{len(fetch_tools)} Fetch + {len(github_tools)} GitHub + {len(docs_tools)} Docs = {len(all_tools)} tools")
print([t.name for t in all_tools])

1 Fetch + 19 GitHub + 2 Docs = 22 tools
['fetch', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories', 'search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain']


In [14]:
# Tools from three servers + the resource (guide_text, Step 5) in the system prompt
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You help an AI consultant shortlist agent frameworks for a client RFP. "
        "Use the GitHub tools for repository facts: the license from the LICENSE file, and the latest release. "
        "Use the LangChain docs tools for anything about LangChain or LangGraph. "
        "Use the fetch tool for other web pages, such as the OpenAI Agents SDK docs. "
        "Never guess: every fact must come from a tool result or from the guide below, "
        "and name the repo, file, link or URL behind it. "
        "If neither gives a fact, write 'not found in the sources'.\n\n"
        f"GUIDE (the LangChain skill guide, an MCP resource):\n{guide_text}"
    ),
)

question = (
    "We are shortlisting agent frameworks for a client RFP. "
    "For LangGraph (GitHub repo langchain-ai/langgraph) and the OpenAI Agents SDK (GitHub repo openai/openai-agents-python): "
    "give each one's license and latest release from GitHub, and one sentence on what each framework is for: "
    "for LangGraph from the LangChain docs, for the OpenAI Agents SDK from https://openai.github.io/openai-agents-python/. "
    "Answer as a small table with a source column. "
    "Below the table, add one line from the guide: when should a team choose LangGraph instead of LangChain's create_agent?"
)

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
combined_answer = combined_result["messages"][-1].content
print("\nAnswer:\n", combined_answer)

# Evidence for the guide line: a resource leaves no [LOG] line, so check key words again (as in Step 5)
for word in ["deterministic", "human-in-the-loop"]:
    print(f"'{word}': in answer = {word in combined_answer.lower()}, in guide = {word in guide_text.lower()}")

[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG] github -> get_latest_release  {'owner': 'openai', 'repo': 'openai-agents-python'}
[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_file_contents  {'owner': 'openai', 'repo': 'openai-agents-python', 'path': 'LICENSE'}
[LOG]    returned: {"tag_name":"1.2.18","target_commitish":"3521f08e21a7d89c28331191374bc85c11ecb6e2","name":"langgraph==1.2.18","body":"Changes since 1.2.17\n\n* releas...
[LOG]    returned: {"tag_name":"v0.24.0","target_commitish":"17d994bd3045dae71acda676a4ef4762ed8ecd94","name":"v0.24.0","body":"## [0.24.0](https://github.com/openai/ope...
[LOG]    returned: successfully downloaded text file (SHA: fc0602feecdd6748623c852ab534e1ca612673c7) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...
[LOG]    returned: successfully downloaded text file (SHA: e5ad2c5aa30343f143a5c418004ca1a8d2cfaff4) ty

**Done when:** the `[LOG]` lines show calls to **all three** servers (`fetch`, `github`, `langchain-docs`), each fact in the table matches one of them, and the guide line passes the keyword check.

### One question of our own: a failure probe

Pricing is **not** in our sources: not in the repos, not on the OpenAI Agents SDK page. A grounded agent must say so instead of inventing a number. The LangChain docs are now a tool too, and they also cover **LangSmith**, LangChain's paid platform: watch whether the agent presents LangSmith plans as *LangGraph* support. That would be a plausible but unsupported answer.

In [15]:
question = (
    "Does LangGraph or the OpenAI Agents SDK offer paid enterprise support, and what does it cost per year? "
    "Use only your tools."
)

own_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:\n", own_result["messages"][-1].content)

[LOG] langchain-docs -> search_docs_by_lang_chain  {'query': 'LangGraph enterprise support pricing'}
[LOG] fetch -> fetch  {'url': 'https://platform.openai.com/docs/guides/agents'}
[LOG]    returned: Title: LangSmith for Enterprise
Link: https://docs.langchain.com/langsmith/enterprise
Page: langsmith/enterprise
Content: This page is a reference hub...
[LOG]    returned: Contents of https://platform.openai.com/docs/guides/agents:
<p>Agents | OpenAI API</p>

For the complete documentation index, see llms.txt. Markdown v...
[LOG] fetch -> fetch  {'url': 'https://www.langchain.com/pricing'}
[LOG] fetch -> fetch  {'url': 'https://www.langchain.com/contact-sales'}
[LOG]    returned: Contents of https://www.langchain.com/contact-sales:
<p>Contact the LangChain Sales Team</p>

Try LangSmith

Get a demo

# See LangSmith in action

La...
[LOG]    returned: Contents of https://www.langchain.com/pricing:
<p>LangSmith Plans and Pricing</p>

Try LangSmith

Get a demo

Pricing

# Plans for teams of a

### Before you submit: secrets check

**Save the notebook first** (Cmd+S): the check reads the saved file, including all outputs. It prints only PASS or FAIL, never a key.

In [16]:
import re
import subprocess

# 1. .env must be git-ignored, so it can never be committed
ignored = subprocess.run(["git", "check-ignore", "-q", ".env"]).returncode == 0
print("PASS" if ignored else "FAIL", ".env is git-ignored")

# 2. No OpenAI key or GitHub token anywhere in this notebook (code or outputs)
notebook_text = open("mcp_langchain_3servers.ipynb", encoding="utf-8").read()
secret_pattern = r"\bsk-[A-Za-z0-9_-]{20,}|github_pat_[A-Za-z0-9_]{20,}|ghp_[A-Za-z0-9]{20,}"
found = len(re.findall(secret_pattern, notebook_text))
print("PASS" if found == 0 else f"FAIL ({found} found: clear the outputs)", "no keys or tokens in the notebook")

PASS .env is git-ignored
PASS no keys or tokens in the notebook
